# Часть B. Окружения Python

Перед запуском выберите GPU: Runtime → Change runtime type → L4 (для этой части её хватает).
Подробный разбор каждой строки — в файле [docs/A_B_explained.md](https://github.com/IvanovskyDev/Machine-Unlearning-in-LLM/blob/main/docs/A_B_explained.md).

**1. Старт сессии** — то же, что шаги 1–4 части A, плюс папка `envs` на Drive для lock-файлов.

In [ ]:
import os                                   # папки и переменные окружения

from google.colab import drive, userdata    # Google Drive и секреты Colab

drive.mount("/content/drive")               # подключить Google Drive

DRIVE = "/content/drive/MyDrive/unlearning_data"   # папка проекта на Drive (постоянная)
FAST = "/content/fast"                             # папка на диске машины (очищается после сессии)

os.makedirs(DRIVE + "/saves", exist_ok=True)       # чекпоинты моделей
os.makedirs(DRIVE + "/results_raw", exist_ok=True) # сырые результаты атак
os.makedirs(DRIVE + "/envs", exist_ok=True)        # lock-файлы окружений
os.makedirs(FAST + "/hf_home", exist_ok=True)      # кэш Hugging Face
os.makedirs(FAST + "/models", exist_ok=True)       # скачанные модели

os.environ["BIG"] = DRIVE                          # «большой диск» из плана
os.environ["HF_HOME"] = FAST + "/hf_home"          # кэш Hugging Face
os.environ["MODELS"] = FAST + "/models"            # папка моделей
os.environ["TOKENIZERS_PARALLELISM"] = "false"     # меньше лишних предупреждений
os.environ["PYTHONUNBUFFERED"] = "1"               # вывод программ сразу попадает в лог
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # токен Hugging Face из секрета HF_TOKEN

print("Старт сессии выполнен")

**2. Ставим uv** — установщик пакетов вместо conda: он создаёт отдельные окружения со своим Python. Сначала ячейка убирает настройки Colab, которые мешают нашим окружениям.

In [ ]:
# Colab задаёт свои переменные окружения, которые мешают нашим окружениям:
#   UV_...      — велят uv ставить пакеты в системный Python 3.13 Colab с его ограничениями версий;
#   PYTHONPATH  — подмешивает модули Colab в любой запущенный Python;
#   MPLBACKEND  — настройка графиков блокнота; в наших окружениях из-за неё падают vLLM и BERTScore.
# Цикл проходит по всем переменным и удаляет эти; os.environ.pop удаляет переменную и возвращает её значение.
for name in list(os.environ):
    if name.startswith("UV_") or name in ["PYTHONPATH", "MPLBACKEND"]:
        print("Убираю", name, "=", os.environ.pop(name))

# поставить uv (-q — без подробного вывода) и проверить, что он работает
!pip install -q uv
!uv --version

**3. Скачиваем OpenUnlearning** ровно той версии, с которой сверен план (коммит `4ad738a`), и связываем его папку `saves` с Drive. Должна напечататься строка `4ad738a … Mar 18 … 2026`.

In [ ]:
%%bash
set -e                                   # остановиться на первой ошибке
rm -rf /content/sandbox/open-unlearning  # удалить старую копию, если есть (данные на Drive не трогаются)
mkdir -p /content/sandbox                # создать папку-«песочницу»
cd /content/sandbox                      # перейти в неё
git clone -q https://github.com/locuslab/open-unlearning.git     # скачать репозиторий OpenUnlearning
cd open-unlearning
git checkout -q 4ad738aaf60f6a4385f6e2506d01da99e76c31f3         # переключиться на закреплённую версию
git log -1 --format='%h %cd'             # напечатать короткий номер версии и её дату
ln -s $BIG/saves saves                   # ссылка saves → папка на Drive: результаты сразу попадают туда
ls -l saves                              # показать, куда ведёт ссылка

**4. Создаём окружение `unl` и ставим зависимости OpenUnlearning** (Python 3.11, torch 2.4.1, transformers 4.51.3). Займёт несколько минут. В выводе должно быть `Using Python 3.11… environment at: /content/envs/unl`.

In [ ]:
%%bash
set -e
uv venv /content/envs/unl --python 3.11 --seed --clear   # создать окружение unl с Python 3.11 (--clear — заменить старое)
source /content/envs/unl/bin/activate                    # «войти» в окружение: следующие команды работают в нём
cd /content/sandbox/open-unlearning
# зависимости OpenUnlearning (requirements.txt) и lm-eval для MMLU;
# setuptools<81 — в более новых версиях нет модуля pkg_resources, без него не запускается TensorBoard 2.18
uv pip install ".[lm-eval]" "setuptools<81"

**5. Ставим FlashAttention-2** — готовую сборку под Python 3.11, torch 2.4 и CUDA 12, поэтому без часовой компиляции.

In [ ]:
%%bash
set -e
source /content/envs/unl/bin/activate
# готовый файл FlashAttention 2.6.3: cp311 — Python 3.11, torch2.4 — torch 2.4, cu123 — CUDA 12
uv pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"

**6. Скачиваем логи оценки готовых моделей** — они нужны для Forget Quality. Файлы попадают в `saves/eval`, то есть на Drive; в конце печатается список папок `tofu_…`.

In [ ]:
%%bash
set -e
source /content/envs/unl/bin/activate
cd /content/sandbox/open-unlearning
python setup_data.py --eval_logs         # скачать логи оценки в saves/eval (то есть на Drive)
ls saves/eval                            # показать скачанные папки

**7. Проверяем `unl`.** Должно напечататься `2.4.1+cu121 12.1 True <ваша GPU>` и `4.51.3 2.6.3`.

In [ ]:
%%bash
source /content/envs/unl/bin/activate
# python -c "..." — выполнить код Python, записанный в кавычках:
# версии torch и CUDA, видна ли GPU и её имя; затем версии transformers и FlashAttention
python -c "
import torch, transformers, flash_attn
print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))
print(transformers.__version__, flash_attn.__version__)
"

**8. Записываем lock-файл `unl`** — точные версии всех пакетов, файл `envs/requirements-unl.lock` на Drive. FlashAttention и сам OpenUnlearning в него не входят: они ставятся отдельно.

In [ ]:
%%bash
set -e
source /content/envs/unl/bin/activate
# uv pip freeze — список всех пакетов с точными версиями;
# | передаёт его команде grep -v, которая выбрасывает строки с flash-attn и open-unlearning;
# > записывает результат в файл на Drive
uv pip freeze | grep -v -e flash-attn -e open-unlearning > $BIG/envs/requirements-unl.lock
wc -l $BIG/envs/requirements-unl.lock    # сколько строк (пакетов) записано

**9. Создаём окружение `atk` и ставим vLLM, оценщик и пакеты для анализа.** Займёт несколько минут. В конце должно напечататься `All installed packages are compatible`.

In [ ]:
%%bash
set -e
uv venv /content/envs/atk --python 3.11 --seed --clear   # отдельное окружение atk с Python 3.11
source /content/envs/atk/bin/activate
# vLLM 0.19.1 — последняя версия под CUDA 12: работает и на драйверах с CUDA 12, и с CUDA 13.
# transformers, tokenizers и huggingface-hub — ровно те версии, с которыми тестировался vLLM 0.19.1.
# setuptools<81 — как в тестах vLLM: в более новых версиях нет модуля pkg_resources.
# cupy-cuda12x (spaCy на GPU) ставится отдельно, а не через spacy[cuda12x]: иначе uv откатит vLLM к старой версии.
# Последняя строка — модель spaCy en_core_web_trf 3.8.0 (ту же ставит команда spacy download).
# \ в конце строки — команда продолжается на следующей строке.
uv pip install "vllm==0.19.1" "transformers==5.5.3" "tokenizers==0.22.2" "huggingface-hub==1.10.2" "setuptools<81" \
    openai spacy cupy-cuda12x bert-score sentence-transformers rapidfuzz \
    pandas pyarrow orjson zstandard hydra-core tenacity \
    scipy statsmodels scikit-learn lifelines matplotlib \
    pytest hypothesis ruff mypy \
    "en_core_web_trf @ https://github.com/explosion/spacy-models/releases/download/en_core_web_trf-3.8.0/en_core_web_trf-3.8.0-py3-none-any.whl"
uv pip check                             # проверить, что версии всех пакетов совместимы между собой

**10. Записываем скрипт проверки `atk`** в файл `/content/check_atk.py`; запускается он в шаге 11.

In [ ]:
%%writefile /content/check_atk.py
# Проверка окружения atk (запускается в шаге 11)

# 1) PyTorch видит GPU, vLLM импортируется
import torch
import vllm

print("torch", torch.__version__, "| CUDA доступна:", torch.cuda.is_available(), "| vLLM", vllm.__version__)

# 2) spaCy работает на GPU и находит в тексте имя, место и дату
import spacy

print("spaCy на GPU:", spacy.prefer_gpu())
nlp = spacy.load("en_core_web_trf")
doc = nlp("Basil Mahfouz Al-Kuwaiti was born in Kuwait City in 1956.")
for entity in doc.ents:
    print("   ", entity.text, "—", entity.label_)

# 3) BERTScore сравнивает два предложения по смыслу; F1 — одно число, чем больше, тем ближе смысл
from bert_score import BERTScorer

scorer = BERTScorer(lang="en", model_type="roberta-large", rescale_with_baseline=True)
precision, recall, f1 = scorer.score(["He was born in Kuwait."], ["The author was born in Kuwait City."])
print("BERTScore F1:", round(f1.item(), 3))

**11. Проверяем `atk`.** Должно быть: версия vLLM, `CUDA доступна: True`, `spaCy на GPU: True`, сущности с метками PERSON, GPE и DATE и одно число BERTScore F1. При первом запуске BERTScore скачивает модель roberta-large (около 1,4 ГБ).

In [ ]:
%%bash
source /content/envs/atk/bin/activate
vllm --version                           # версия vLLM из командной строки
python /content/check_atk.py             # запустить проверку из шага 10

**12. Записываем lock-файл `atk`** — файл `envs/requirements-atk.lock` на Drive.

In [ ]:
%%bash
set -e
source /content/envs/atk/bin/activate
uv pip freeze > $BIG/envs/requirements-atk.lock   # точные версии всех пакетов atk — в файл на Drive
wc -l $BIG/envs/requirements-atk.lock             # сколько пакетов записано

**13. Записываем окружения в журнал** (`journal.md` на Drive). Выполняйте один раз после каждой сборки окружений.

In [ ]:
from datetime import datetime    # текущие дата и время
from zoneinfo import ZoneInfo    # часовые пояса

# grep -E "^(...)==" — найти в lock-файле строки, которые начинаются с названий этих пакетов
unl = !grep -E "^(torch|transformers|accelerate|deepspeed)==" $BIG/envs/requirements-unl.lock
atk = !grep -E "^(torch|vllm|transformers|spacy|cupy-cuda12x)==" $BIG/envs/requirements-atk.lock
unl_versions = ", ".join(unl)    # склеить найденные строки через запятую
atk_versions = ", ".join(atk)
today = datetime.now(ZoneInfo("Europe/Moscow")).date()   # дата по Москве

text = f"""
## {today} — Окружения unl и atk (Colab)
- OpenUnlearning: коммит 4ad738a
- unl: Python 3.11, {unl_versions}, flash-attn 2.6.3
- atk: Python 3.11, {atk_versions}
- Lock-файлы: {DRIVE}/envs/requirements-unl.lock и requirements-atk.lock
"""

with open(DRIVE + "/journal.md", "a", encoding="utf-8") as f:   # дописать запись в конец журнала
    f.write(text)

print(text)

**14. Проверяем, что окружения собираются из lock-файлов** (блок 9 плана). Runtime → Disconnect and delete runtime, затем выполните шаги 1 и 2, эту ячейку и проверки 7, 10 и 11. Так же окружения будут собираться в начале следующих частей.

In [ ]:
%%bash
set -e
# unl: новое окружение, пакеты ровно по lock-файлу (-r — список из файла), затем FlashAttention (его в lock-файле нет)
uv venv /content/envs/unl --python 3.11 --seed --clear
source /content/envs/unl/bin/activate
uv pip install -r $BIG/envs/requirements-unl.lock
uv pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
deactivate                               # «выйти» из окружения unl

# atk: новое окружение и пакеты ровно по lock-файлу
uv venv /content/envs/atk --python 3.11 --seed --clear
source /content/envs/atk/bin/activate
uv pip install -r $BIG/envs/requirements-atk.lock
uv pip check